In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from eucare.base import *
import eucare.plotting as euplot


plt.figure(figsize=(5, 5))
prev_poly = regular_poly_points(3)
for i in range(3, 20):
    poly = regular_poly_points(i) * np.random.rand()
    j = np.random.randint(0, i-2)
    mat = find_affine(poly[j:j+2][::-1], prev_poly[i-3:i-1])
    poly = apply_affine(poly, mat)
    euplot.plot_polygon(poly)
    prev_poly=poly
    #plt.scatter(*(poly[j:j+2].T))
    
euplot.set_equal_aspect()

## HalfEdge Data Structure

In [ ]:
from eucare.half import AttributeObject

a = AttributeObject()

print(a.has_attributes())
a['adsf'] = 'party'
print(a.has_attributes())
for key, val in a.items():
    print(key, val)


In [ ]:
from eucare.half import Vertex, CyclicHalfedgeGraph, IdObject
from tqdm import tqdm_notebook as tqdm
import networkx as nx

IdObject.reset_ids()
poly = CyclicHalfedgeGraph([Vertex() for i in range(4)])
#for v in poly.vertices:
#    print(v)
#    for h in v.outgoing_iter():
#        pass
#        print(h.orig, h.dest)

#hs = list(poly.halfedges)

#f = any_element(poly.faces)
#print(f.__dict__)
#[print(v) for v in f.reverse_halfedge_iter()]


for i in range(3):
    print(i, poly.order)
    border_vertices = poly.border_vertices()
    for h1 in tqdm(list(poly.border_edge_iter())):
        #print(h1)
        to_attach = CyclicHalfedgeGraph([Vertex() for i in range(5)])
        h2 = to_attach.get_any_border()
        poly.add_graph(to_attach)
        poly.glue_e2e(h1, h2)
    for v in border_vertices:
        poly.close_vertex(v)

poly.show_spring_layout()

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
from eucare.half import HalfEdgeGraph, Vertex, IdObject, RegularNGon, CyclicHalfedgeGraph, InAngleHEG, EuclideanPositionHEG
from eucare.instructions import *
from eucare.base import unit_vector
from copy import deepcopy, copy
from tqdm import tqdm_notebook as tqdm
from eucare.plotting import plot_polygon
import matplotlib.pyplot as plt

    
#tile = copy(tile)

#def RegularNGon(n):
#    return CyclicHalfedgeGraph([Vertex() for _ in range(n)])

#n = 6
#proto_tile = RegularEuclideanTile(n, edge_labels = ['a'] * n)

def print_graph(graph):
    for e in graph.halfedges:
        print(e.__repr__(), e.on_border(), e.face)
        
def attatch_tile_instruction(proto_tile, label=None):
    def instruction(graph, edge):
        tile, edge_dict = proto_tile.make_graph()
        if label is not None:
            edge = edge_dict[label]
        else:
            # just take any edge
            edge = next(iter(edge_dict.values()))
        graph.glue_graph_e2e(tile, e, edge)
    return instruction

# define 6.4.3.4 tiling
hexagon = RegularEuclideanTile(6, edge_labels=['a', 'a', 'a', 'a', 'a', 'a'])
square = RegularEuclideanTile(4, edge_labels=['b', 'c', 'b', 'c'])
triangle = RegularEuclideanTile(3, edge_labels=['d', 'd', 'd'])
hexagon.edge_instructions['a'] = attatch_tile_instruction(square, 'b')
square.edge_instructions['b'] = attatch_tile_instruction(hexagon)
square.edge_instructions['c'] = attatch_tile_instruction(triangle)
triangle.edge_instructions['d'] = attatch_tile_instruction(square, 'c')
        

#tile = RegularNGon(n)
#print_graph(tile)
#for e in tile.border_edge_iter():
#    e['instruction'] = attatch_tile_instruction(proto_tile, 'a')
    

#for e in tile1.border_edge_iter():
#    e['instruction'] = instruction0
  

# possible problem: topology based merging can miss geometry

for k in tqdm(range(1)):
    IdObject.reset_ids()
    tiling = EuclideanPositionHEG(eps=1e-3, other=hexagon.make_graph(add_positions=True)[0])
    for i in range(1):
        for e in tiling.border_edges():
            if e.on_border() and e in tiling.halfedges:
                tiling.execute_edge_instruction(e)
                    
                #print('...',tiling.order,'...')

print('drawing layout')

for face in tiling.faces:
    points = np.stack([v['pos'] for v in face.vertex_iter()])
    plot_polygon(points)
plt.show()
#tiling.show_spring_layout()

In [ ]:
from eucare.base import unit_vector, angle_to_axis
unit_vector(np.pi/2)

In [ ]:
(8.37758 - 6.28318) / np.pi

In [ ]:
import numpy as np
import collections

class EuclideanVertex2D(Vertex):
    def __init__(self, pos, any_outgoing=None):
        super(EuclideanVertex2D, self).__init__(any_outgoing)
        if not isinstance(pos, colledctions.Sized):
            raise ValueError(f"position must be Sized. Got {pos}.")
        if len(pos) != 2:
            raise ValueError(f"Got position of length {len(pos)} != 2.")
        self.pos = np.array(pos, dtype=np.float32)
    
    @property
    def x(self):
        return self.pos[0]
    
    @property
    def y(self):
        return self.pos[1]

In [ ]:
v1, v2 = Vertex(), Vertex()
h1, h2 = HalfEdge(orig=v1, dest=v2), HalfEdge(orig=v2, dest=v1)
h1.rev = h2
h2.rev = h1
e = Edge(h1, h2)

In [ ]:
e[h1.orig]

In [ ]:
from copy import copy

v = Vertex()
d = dict()
d[v] = 3
v.any_outgoing = 123
d[v]

In [ ]:
from eucare.half import Vertex
from copy import deepcopy
a = Vertex()
a['a'] = a
b = deepcopy(a)
b['a']